# OpenFOAM trên Colab G4 · chạy tự động, không lưu gì

Notebook này nhận job do Mac Mini M4 đẩy lên GitHub và chạy OpenFOAM 14 với toàn bộ phần cứng của runtime G4.
Mọi thứ diễn ra trong RAM (`/dev/shm`); không ghi tệp log, không mount Drive, không đẩy kết quả lên đâu.
Kết quả chỉ hiện ở ô bên dưới.

**Làm một lần**

1. *Runtime → Change runtime type* → chọn **G4** (NVIDIA RTX PRO 6000).
2. *Edit → Notebook settings* → tick **Omit code cell output when saving this notebook** (để đầu ra không bị lưu).
3. Biểu tượng chìa khoá (*Secrets*) → thêm `GH_REPO` = `owner/tên-repo`; với repo riêng tư thêm `GH_TOKEN`
   (fine-grained token, chỉ quyền **Contents: Read-only** trên đúng repo đó). Tuỳ chọn: `G4_BRANCH` (mặc định `main`),
   `G4_SUBDIR` (thư mục bộ công cụ trong repo, mặc định `openfoam-g4`; để trống nếu bộ công cụ nằm ở gốc repo),
   `G4_ALLOW_MODELS` = `1` nếu muốn chạy mô hình Python của bạn trên GPU khi OpenFOAM không dùng GPU.

**Mỗi phiên**: *Runtime → Run all*. Ô cuối cùng theo dõi `jobs/active.json` mỗi 30 giây và chạy job mới.
CPU và GPU chạy song song: solver bắt đầu ngay trên mọi lõi, OGL được dựng nền, xong thì phần giải áp suất
chuyển sang GPU nếu đo thấy nhanh hơn; lúc GPU rảnh, làn mô hình (nếu bật) dùng nó.
Với Colab Pro+, chọn chạy nền để phiên tiếp tục tới 24 giờ khi đóng trình duyệt.

In [ ]:
# 1 · Kiểm tra runtime: phải thấy RTX PRO 6000 và 96 GB VRAM
import os, subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(out.stdout.strip() or "KHÔNG CÓ GPU: chọn lại runtime G4 nếu muốn dùng GPU (CPU vẫn chạy được)")
print("vCPU:", os.cpu_count())

In [ ]:
# 2 · Đọc cấu hình từ Colab Secrets vào biến môi trường của phiên (không in ra, không ghi đĩa)
import os, getpass

def secret(name, default=""):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return default

os.environ["G4_REPO"] = secret("GH_REPO") or input("GitHub repo (owner/name): ").strip()
token = secret("GH_TOKEN") or getpass.getpass("GitHub token (Enter nếu repo công khai): ")
if token:
    os.environ["G4_GIT_TOKEN"] = token
del token
os.environ["G4_BRANCH"] = secret("G4_BRANCH", "main")
os.environ["G4_SUBDIR"] = secret("G4_SUBDIR", "openfoam-g4")
# chỉ bật khi bạn muốn chạy mô hình Python của chính mình trên GPU lúc OpenFOAM không dùng GPU
os.environ["G4_ALLOW_MODELS"] = "1" if secret("G4_ALLOW_MODELS") == "1" else ""
print("repo:", os.environ["G4_REPO"], "· nhánh:", os.environ["G4_BRANCH"],
      "· thư mục:", os.environ["G4_SUBDIR"] or "(gốc repo)",
      "· token:", "có" if os.environ.get("G4_GIT_TOKEN") else "không",
      "· làn mô hình:", "bật" if os.environ["G4_ALLOW_MODELS"] else "tắt")

In [ ]:
# 3 · Lấy bộ chạy vào RAM rồi giao toàn bộ việc cho nó: đo phần cứng, cài OpenFOAM 14,
#     theo dõi job, tự khai thác hết lõi CPU / GPU / RAM, tự chữa lỗi, in kết quả.
import os, stat, subprocess, sys

ram = "/dev/shm/g4"
os.makedirs(ram, mode=0o700, exist_ok=True)
askpass = os.path.join(ram, "askpass.sh")          # không chứa bí mật: đọc token từ biến môi trường
with open(askpass, "w") as fh:
    fh.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) printf "%s\\n" "$G4_GIT_TOKEN" ;; esac\n')
os.chmod(askpass, stat.S_IRWXU)
env = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0", PYTHONDONTWRITEBYTECODE="1")

boot = os.path.join(ram, "boot")
if not os.path.isdir(os.path.join(boot, ".git")):
    subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--depth", "1", "--single-branch",
                    "--branch", os.environ["G4_BRANCH"], f"https://github.com/{os.environ['G4_REPO']}.git", boot],
                   check=True, env=env)
pkg = os.path.join(boot, os.environ["G4_SUBDIR"]) if os.environ["G4_SUBDIR"] else boot
env["PYTHONPATH"] = pkg

cmd = [sys.executable, "-B", "-m", "g4runner", "colab-up", "--repo", os.environ["G4_REPO"],
       "--branch", os.environ["G4_BRANCH"], "--subdir", os.environ["G4_SUBDIR"], "--watch", "--interval", "30"]
proc = subprocess.Popen(cmd, cwd=pkg, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("bộ chạy dừng, mã thoát", proc.wait())